
# 漢字・アルファベットからカタカナを推測するモデルを学習する
オープンソース辞書及び，Wikipediaから作成した漢字とカタカナのペアデータを用いて，Transformer+Seq2Seqのモデルで，スクラッチで学習する。

入力の文字（漢字・アルファベットなど）は一文字ずつに分割，また，出力の文字（カタカナ）も一文字>ずつに分割して，学習する。



## データ準備
データは，[dict](../dict)で作成した姓名辞書データのoss.json及びseimei.jsonと，Wikipediaから抜き出した，漢字姓名とカタカナ姓名（スペース区切りのもの）を用いて，学習に使用した。単語単位である姓名辞書データと，Wikipediaのスペースで区切られた姓名を用いることで，姓名で漢字とカナが入力された場合の学習を可能とした。

また，ひらがなとカタカナ，


In [1]:
import os
ver="2.0o"

os.environ["VER"]=ver
os.environ['DATA']='../dict/data/data.'+os.environ["VER"]
os.environ['DIR']='dataset/dataset.'+os.environ["VER"]
os.environ['LOG']='logs/logs.'+os.environ["VER"]
os.environ['MODEL']='model/model.'+os.environ["VER"]

In [2]:
!echo $VER

2.0o


In [3]:

!mkdir -p $DIR
!mkdir -p $LOG
!mkdir -p $MODEL

In [5]:
# オープンソース辞書とWikipediaから作成した単語辞書をマージする
!python mergedic.py --dic1 $DATA/reliable_$VER.json --dic2 $DATA/tankanji_$VER.json --outfile tmp.json

# 作成したデータを訓練用，開発用，検証用に分ける。開発用と検証用がそれぞれ全体の0.01
!python prep.py --json tmp.json --outdir $DIR --ratio 0.01

# ひらがなとカタカナのペアデータを作成し，訓練データに混ぜる 
!python kana.py --outfile $DIR/hirakata.txt

!cat $DIR/hirakata.txt |awk -F, '{print $1}'>> $DIR/train.src
!cat $DIR/hirakata.txt |awk -F, '{print $2}'>> $DIR/train.tgt

# 単漢字とカタカナのデータを，訓練データに混ぜる
!python json2txt.py --json $DATA/tankanji_$VER.json --txt tmp.txt

!cat tmp.txt |awk -F, '{print $1}' >> $DIR/train.src
!cat tmp.txt |awk -F, '{print $2}' >> $DIR/train.tgt
!rm -rf tmp.txt tmp.json

# 開発，検証用に，訓練データに入っているものがあれば，削除する(インサンプルを阻止)
!python omit_testval.py --train_src $DIR/train.src --train_tgt $DIR/train.tgt --test_src $DIR/test.src --test_tgt $DIR/test.tgt --valid_src $DIR/valid.src --valid_tgt $DIR/valid.tgt

# 訓練，開発，検証データをシャッフルする
!python shuffle.py --src $DIR/train.src --tgt $DIR/train.tgt
!python shuffle.py --src $DIR/valid.src --tgt $DIR/valid.tgt
!python shuffle.py --src $DIR/test.src --tgt $DIR/test.tgt

# スペース区切りにする
!python space.py --infile $DIR/train.src --outfile $DIR/train.src
!python space.py --infile $DIR/train.tgt --outfile $DIR/train.tgt
!python space.py --infile $DIR/valid.src --outfile $DIR/valid.src
!python space.py --infile $DIR/valid.tgt --outfile $DIR/valid.tgt
!python space.py --infile $DIR/test.src --outfile $DIR/test.src
!python space.py --infile $DIR/test.tgt --outfile $DIR/test.tgt






{
  "dic1": "../dict/data/data.2.0o/reliable_2.0o.json",
  "dic2": "../dict/data/data.2.0o/tankanji_2.0o.json",
  "outfile": "tmp.json"
}
len=1720809
cnt=1851768
{
  "json": "tmp.json",
  "outdir": "dataset/dataset.2.0o",
  "ratio": 0.01
}
k_max_len=72
v_max_len=120
train,1814734
valid,18517
test,18517
{
  "outfile": "dataset/dataset.2.0o/hirakata.txt"
}
{
  "json": "../dict/data/data.2.0o/tankanji_2.0o.json",
  "txt": "tmp.txt"
}
{
  "train_src": "dataset/dataset.2.0o/train.src",
  "train_tgt": "dataset/dataset.2.0o/train.tgt",
  "test_src": "dataset/dataset.2.0o/test.src",
  "test_tgt": "dataset/dataset.2.0o/test.tgt",
  "valid_src": "dataset/dataset.2.0o/valid.src",
  "valid_tgt": "dataset/dataset.2.0o/valid.tgt"
}
train=1932855
omit=2359
{
  "tgt": "dataset/dataset.2.0o/train.tgt",
  "src": "dataset/dataset.2.0o/train.src"
}
{
  "tgt": "dataset/dataset.2.0o/valid.tgt",
  "src": "dataset/dataset.2.0o/valid.src"
}
{
  "tgt": "dataset/dataset.2.0o/test.tgt",
  "src": "dataset/dataset.

In [6]:
# format
!python format.py --src $DIR/train.src --tgt $DIR/train.tgt --outfile $DIR/train.jsonl
!python format.py --src $DIR/test.src --tgt $DIR/test.tgt --outfile $DIR/test.jsonl
!python format.py --src $DIR/valid.src --tgt $DIR/valid.tgt --outfile $DIR/valid.jsonl



In [7]:
!date

2026年  4月  8日 水曜日 16:06:50 JST
